# Run basic calculations using Workgraph

## Aim

The workgraph demonstrates combining a gemotery optimisation followed by a phonon calculation to obtain force constants. Both employ Janus-core

### Setup

The initial setup is very similar to the other tutorials, such as `singlepoint.ipynb`, which goes into more detail about what each step is doing. For simplicity all the imports are done at the start.

In [ ]:
from aiida.plugins import CalculationFactory
from aiida import load_profile
from aiida_mlip.data.model import ModelData
from aiida.orm import StructureData
from aiida.orm import load_code
from aiida.orm import Str, Float, Bool, Int

from aiida_workgraph import WorkGraph

from ase.build import bulk
from ase.io import read

Load the aiida profile:

In [ ]:
# Load profile
load_profile()

Get the structure, model and load the code. NaCl is being used for the structure (note the lattice parameter).

In [ ]:
#FCC NaCl with 5.63 lattice parameter. This gives 2.815 in the reduced unit cell.
structure = StructureData(ase=bulk("NaCl", "rocksalt", 5.63))

In [ ]:

#from url
uri = "https://github.com/stfc/janus-core/raw/main/tests/models/mace_mp_small.model"
model = ModelData.from_uri(uri, architecture="mace", cache_dir="mlips")

#or from file
model = ModelData.from_local("/path/to/model", architecture="mace")

In [ ]:
code = load_code("janus@localhost")

Inputs should include the model, code, metadata, and any other keyword arguments expected by the calculation we are running. The input for the geometry optimisation is completed first followed by the inputs for the phonon calculation.

In [ ]:
#inpits for the geometry optimisation
inputs_geom = {
        "code": code,
        "model": model,
        "struct": structure,
        "arch": Str(model.architecture),
        "device": Str("cpu"),
        "fmax": Float(0.1),
        "opt_cell_lengths": Bool(True),
        "opt_cell_fully": Bool(True),
        "metadata": {"options": {"resources": {"num_machines": 1}}},
}


In [ ]:
#input for phonon.
inputs_phon = {
        "metadata": {"options": {"resources": {"num_machines": 1}}},
        "code": code,
        "arch": model.architecture,
        "model": model,
        "device": Str("cpu"),
        "supercell": Str("2 2 2"),
        "displacement": Float(0.01),
        "nqpoints": Int(51),
        "dos": Bool(False),
        "pdos": Bool(False),
        "bands": Bool(False),
        "no_hdf5": Bool(False),
        "symmetrize": Bool(False),
}


Instantiate both geometry optimisation and phonon calculations.

In [ ]:
geomoptCalc = CalculationFactory("mlip.opt")
phononCalc = CalculationFactory("mlip.ph")

### Creating and running a workgraph

We can now create a workgraph by first loading WorkGraph and giving it a name  (`"GeomOptPhonGraph"` in this example). 

In [ ]:
wg = WorkGraph("GeomOptPhonGraph")



We then create a task for our calculation and assign this task a name. The name is then used to retrieve the final structure for use in the phonon calculation.

In [ ]:
gm_calc = wg.add_task(
    geomoptCalc,
    name="geomopt_calc",
    **inputs_geom
)

opt_struct = gm_calc.outputs.final_structure

Create a task for the phonon calculation using the optimised structure.

In [ ]:
ph_calc = wg.add_task(
    phononCalc,
    name="ph_calc",
    struct = opt_struct,
    **inputs_phon,
)


For geometry optimisation we are going to pass outputs to the graph

In [ ]:
wg.outputs.results = wg.tasks.geomopt_calc.outputs.results_dict
wg.outputs.results_file = wg.tasks.geomopt_calc.outputs.xyz_output

In [ ]:
wg

We can finally run the tasks

In [ ]:
wg.run()

### Outputs 
We can then check the output to ensure we are getting the correct output.
#### Geometry Optimisation

In [ ]:
type(wg.outputs.results_file.value)

We can also print the outputs of the calculation. Note the new lattice parameter is approximately 2.77 Angstroms with the `MACE-matpes-r2scan-omat-ft.model` .

In [ ]:
wg.outputs.results.value.get_dict()

#### Phonon Calculation

The results for the phonon calculation were not passed to the graph so the output can be retrieved using the alternate method. Note the results use the optimised lattice parameter for the phonon calculation.

In [ ]:
wg.tasks.ph_calc.outputs.results_dict.value.get_dict()